Before running this notebook:
- Install packages: `pip install langchain langchain-google-genai python-dotenv`
- Get an API key from https://aistudio.google.com
- Create a file titled `.env` in the project directory with the line `GEMINI_API_KEY=<your_api_key_here>`

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from pathlib import Path
import os
import requests
import json

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
DEFAULT_MODEL = 'gemini-2.5-flash'
DEFAULT_TEMPERATURE = 0.5
SYSTEM_MODEL = 'gemini-2.5-pro'
SYSTEM_TEMPERATURE = 0.25
TOOL_DATA_DIR = Path('tool_data')
AGENT_REGISTRY_PATH = Path('agents.json')

def make_llm(model=None, temperature=None):
    llm = ChatGoogleGenerativeAI(
        google_api_key=GEMINI_API_KEY,
        model=model or DEFAULT_MODEL,
        temperature=temperature or DEFAULT_TEMPERATURE
    )
    return llm

def load_json(path):
    with open(path, 'r') as f:
        obj = json.load(f)
    return obj

In [57]:
@tool
def get_weather(city: str) -> str:
    '''
    Get the current weather for a given city.
    Args:
        city: The name of the city to get weather for (e.g. 'Tokyo', 'Paris')
    Returns:
        A string describing the current weather conditions
    '''
    try:
        # Geocode the city name to coordinates
        geo_url = 'https://geocoding-api.open-meteo.com/v1/search'
        geo_response = requests.get(geo_url, params={
            'name': city,
            'count': 1,
            'language': 'en',
            'format': 'json'
        })
        geo_data = geo_response.json()

        if not geo_data.get('results'):
            return f'Could not find location data for "{city}".'

        location = geo_data['results'][0]
        lat = location['latitude']
        lon = location['longitude']
        country = location.get('country', '')

        # Fetch current weather using coordinates
        weather_url = 'https://api.open-meteo.com/v1/forecast'
        weather_response = requests.get(weather_url, params={
            'latitude': lat,
            'longitude': lon,
            'current': [
                'temperature_2m',
                'relative_humidity_2m',
                'wind_speed_10m',
                'weather_code'
            ],
            'timezone': 'auto'
        })
        weather_data = weather_response.json()
        current = weather_data['current']

        # Decode weather code
        weather_codes = load_json(TOOL_DATA_DIR / 'weather_codes.json')
        condition = weather_codes.get(current['weather_code'], 'Unknown condition')

        return (
            f'Weather in {city}, {country}:\n'
            f'- Condition:   {condition}\n'
            f'- Temperature: {current['temperature_2m']}°C\n'
            f'- Humidity:    {current['relative_humidity_2m']}%\n'
            f'- Wind Speed:  {current['wind_speed_10m']} km/h'
        )

    except Exception as e:
        return f'Error fetching weather for "{city}": {str(e)}'

In [58]:
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    '''
    Convert an amount from one currency to another using live exchange rates.
    Args:
        amount: The amount to convert
        from_currency: The source currency code (e.g. 'USD', 'SGD', 'GBP')
        to_currency: The target currency code (e.g. 'EUR', 'JPY', 'THB')
    Returns:
        A string with the converted amount and exchange rate used
    '''
    try:
        url = 'https://api.frankfurter.app/latest'
        response = requests.get(url, params={
            'from': from_currency.upper(),
            'to': to_currency.upper()
        })

        if response.status_code != 200:
            return f'Error: Could not fetch exchange rate (HTTP {response.status_code}).'

        data = response.json()

        # Check both currencies were recognised
        if 'rates' not in data or to_currency.upper() not in data['rates']:
            return (
                f'Could not find exchange rate for '
                f'"{from_currency.upper()}" to "{to_currency.upper()}". '
                f'Please check the currency codes are valid ISO 4217 codes (e.g. "USD", "JPY").'
            )

        rate = data['rates'][to_currency.upper()]
        converted = round(amount * rate, 2)
        date = data.get('date', 'unknown date')

        return (
            f'Date: {date}\n'
            f'Currency conversion: {amount} {from_currency.upper()} = {converted} {to_currency.upper()}\n'
            f'Exchange rate: 1 {from_currency.upper()} = {rate} {to_currency.upper()}'
        )

    except Exception as e:
        return f'Error converting currency: {str(e)}'

In [59]:
@tool
def calculate_trip_budget(
    destination: str,
    num_days: int,
    num_travellers: int,
    daily_accommodation_per_room: float,
    daily_food_per_person: float,
    daily_activities_per_person: float,
    flight_cost_per_person: float,
    currency: str = 'USD'
) -> str:
    '''
    Calculate a full trip budget breakdown for a group of travellers.
    Args:
        destination: The travel destination (e.g. 'Paris', 'Tokyo')
        num_days: Number of days for the trip
        num_travellers: Number of travellers in the group
        daily_accommodation_per_room: Estimated daily hotel/accommodation cost per room
        daily_food_per_person: Estimated daily food and drink cost per person
        daily_activities_per_person: Estimated daily activities and transport cost per person
        flight_cost_per_person: Return flight cost per person
        currency: Currency code for the budget (default: 'USD')
    Returns:
        A detailed budget breakdown as a formatted string
    '''
    try:
        # Validate inputs
        if num_days <= 0:
            return 'Error: number of days must be greater than 0.'
        if num_travellers <= 0:
            return 'Error: number of travellers must be greater than 0.'
        if any(cost < 0 for cost in [
            daily_accommodation_per_room,
            daily_food_per_person,
            daily_activities_per_person,
            flight_cost_per_person
        ]):
            return 'Error: cost values cannot be negative.'

        # Calculate components
        total_accommodation = daily_accommodation_per_room * num_days
        total_food = daily_food_per_person * num_days * num_travellers
        total_activities = daily_activities_per_person * num_days * num_travellers
        total_flights = flight_cost_per_person * num_travellers

        subtotal = total_accommodation + total_food + total_activities + total_flights

        # Suggest 10% contingency buffer
        contingency = round(subtotal * 0.10, 2)
        grand_total = round(subtotal + contingency, 2)
        per_person_total = round(grand_total / num_travellers, 2)

        return (
            f'Trip Budget Breakdown — {destination} ({num_days} days, {num_travellers} traveller(s))\n'
            f'{'─' * 52}\n'
            f'Flights:        {currency} {total_flights:>10.2f}  ({currency} {flight_cost_per_person:.2f} x {num_travellers} people)\n'
            f'Accommodation:  {currency} {total_accommodation:>10.2f}  ({currency} {daily_accommodation_per_room:.2f}/night x {num_days} nights)\n'
            f'Food:           {currency} {total_food:>10.2f}  ({currency} {daily_food_per_person:.2f}/day x {num_days} days x {num_travellers} people)\n'
            f'Activities:     {currency} {total_activities:>10.2f}  ({currency} {daily_activities_per_person:.2f}/day x {num_days} days x {num_travellers} people)\n'
            f'{'─' * 52}\n'
            f'Subtotal:       {currency} {subtotal:>10.2f}\n'
            f'Contingency:    {currency} {contingency:>10.2f}  (10% buffer)\n'
            f'{'─' * 52}\n'
            f'Grand total:    {currency} {grand_total:>10.2f}\n'
            f'Per person:     {currency} {per_person_total:>10.2f}'
        )

    except Exception as e:
        return f'Error calculating budget: {str(e)}'

In [60]:
@tool
def get_destination_info(country_name: str) -> str:
    '''
    Look up key information about a travel destination country.
    Args:
        country_name: The name of the country to look up (e.g. "Japan", "France", "Thailand")
    Returns:
        A string with key travel-relevant facts about the destination country
    '''
    try:
        url = f'https://restcountries.com/v3.1/name/{country_name}'
        response = requests.get(url, params={'fullText': 'false'})

        if response.status_code == 404:
            return f'Could not find country information for "{country_name}". Please check the spelling.'
        if response.status_code != 200:
            return f'Error fetching destination info (HTTP {response.status_code}).'

        data = response.json()[0]

        common_name = data.get('name', {}).get('common', country_name)
        official_name = data.get('name', {}).get('official', 'N/A')

        region = data.get('region', 'N/A')
        subregion = data.get('subregion', 'N/A')
        capitals = data.get('capital', ['N/A'])
        capital = ', '.join(capitals)

        population = data.get('population', 0)
        population_str = f'{population:,}'

        languages = data.get('languages', {})
        languages_str = ', '.join(languages.values()) if languages else 'N/A'

        currencies = data.get('currencies', {})
        currencies_str = ', '.join(
            f'{info.get('name', code)} ({info.get('symbol', '')})'
            for code, info in currencies.items()
        ) if currencies else 'N/A'

        timezones = data.get('timezones', ['N/A'])
        timezones_str = ', '.join(timezones)

        car_info = data.get('car', {})
        driving_side = car_info.get('side', 'N/A').capitalize()

        idd = data.get('idd', {})
        calling_code = (
            idd.get('root', '') + (idd.get('suffixes', [''])[0])
            if idd else 'N/A'
        )
        landlocked = 'Yes' if data.get('landlocked', False) else 'No'

        return (
            f'Destination Info — {common_name}\n'
            f'{'─' * 52}\n'
            f'- Official Name:  {official_name}\n'
            f'- Capital:        {capital}\n'
            f'- Region:         {subregion}, {region}\n'
            f'- Population:     {population_str}\n'
            f'- Languages:      {languages_str}\n'
            f'- Currency:       {currencies_str}\n'
            f'- Timezones:      {timezones_str}\n'
            f'- Driving Side:   {driving_side}\n'
            f'- Calling Code:   {calling_code}\n'
            f'- Landlocked:     {landlocked}\n'
        )

    except Exception as e:
        return f'Error fetching destination info for "{country_name}": {str(e)}'

In [61]:
@tool
def get_travel_advisory(country_name: str) -> str:
    '''
    Get the travel safety advisory level and key risks for a destination country.
    Args:
        country_name: The name of the country to get the advisory for (e.g. "Thailand", "France")
    Returns:
        A string describing the safety level and key considerations for travellers
    '''
    advisories = load_json(TOOL_DATA_DIR / 'advisories.json')
    key = country_name.lower().strip()
    advisory = advisories.get(key)

    if not advisory:
        return (
            f'No advisory data found for "{country_name}". '
            f'This may be an unlisted destination or a spelling issue. '
            f'Treat as Level 1 — Exercise Normal Precautions — and research locally before travel.'
        )

    risks_str = '\n'.join(f'- {r}' for r in advisory['risks'])
    tips_str = '\n'.join(f'- {t}' for t in advisory['tips'])

    return (
        f'Travel Advisory — {country_name.title()}\n'
        f'{'─' * 52}\n'
        f'Safety Level: Level {advisory['level']} — {advisory['label']}\n\n'
        f'Key Risks:\n{risks_str}\n\n'
        f'Traveller Tips:\n{tips_str}\n'
    )

In [62]:
@tool
def search_attractions(city: str, category: str = 'all') -> str:
    '''
    Search for top attractions and activities in a city, optionally filtered by category.
    Args:
        city: The city to search attractions for (e.g. "Tokyo", "Paris", "Bangkok")
        category: Filter by category — one of "culture", "nature", "food", "adventure", or "all" (default: "all")
    Returns:
        A formatted list of recommended attractions and activities
    '''
    all_attractions = load_json(TOOL_DATA_DIR / 'attractions.json')
    key = city.lower().strip()
    attractions = all_attractions.get(key)

    if not attractions:
        return (
            f'No attractions data found for "{city}". '
            f'Currently supported cities are: {", ".join(c.title() for c in all_attractions.keys())}.'
        )

    # Filter by category
    category = category.lower().strip()
    if category != 'all':
        filtered = [a for a in attractions if a['category'] == category]
        if not filtered:
            valid = '", "'.join(['culture', 'nature', 'food', 'adventure', 'all'])
            return (
                f'No attractions found in "{city}" for category "{category}". '
                f'Valid categories are: "{valid}".'
            )
        attractions = filtered

    lines = [
        f'Top Attractions — {city.title()}'
        + (f' ({category.title()})' if category != 'all' else ''),
        '─' * 52
    ]
    for a in attractions:
        lines.append(f'{a['name']} [{a['category'].title()}]:')
        lines.append(f'- {a['description']}')

    return '\n'.join(lines)

In [63]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage

AGENT_REGISTRY = load_json(AGENT_REGISTRY_PATH)
TOOLS = [get_destination_info, search_attractions, get_travel_advisory, get_weather, calculate_trip_budget, convert_currency]

def get_message_output(msg):
    if not msg.content or type(msg.content) == str:
        return msg.content
    
    content = msg.content[0]
    content_type = content['type']
    output = content[content_type]
    return output

def print_history(messages):
        labels = {
            SystemMessage: ('SYSTEM', '\033[90m'),  # grey
            HumanMessage: ('HUMAN', '\033[94m'),    # blue
            AIMessage: ('AI', '\033[92m'),          # green
            ToolMessage: ('TOOL', '\033[93m'),      # yellow
        }
        no_color = '\033[0m'

        for msg in messages:
            label, color = labels.get(type(msg), ('UNKNOWN', no_color))
            
            if isinstance(msg, ToolMessage):
                header = f'{label} (call_id: {msg.tool_call_id})'
            elif isinstance(msg, AIMessage) and msg.tool_calls:
                tool_names = ', '.join(tc['name'] for tc in msg.tool_calls)
                header = f'{label} [calling tools: {tool_names}]'
            else:
                header = label

            print(f'{color}{header}{no_color}')
            print(f'{color}{get_message_output(msg) or '[no content]'}{no_color}')
            print('-' * 52)


class Agent:
    def __init__(self, agent_key, llm):
        self.agent_key = agent_key
        agent_dict = AGENT_REGISTRY[agent_key]
        self.name = agent_dict['name']
        self.description = agent_dict['description']
        self.system_prompt = agent_dict['system_prompt']
        self.tool_keys = agent_dict['tools']
        self.tools = [t for t in TOOLS if t.name in self.tool_keys]
        self.tool_map = {t.name: t for t in self.tools}
        self.llm = llm.bind_tools(self.tools)

    def run(self, query):
        print(f'\nRunning {self.name}...\n')

        messages = [SystemMessage(self.system_prompt), HumanMessage(query)]
        response = self.llm.invoke(messages)
        messages.append(response)

        while response.tool_calls:
            for tool_call in response.tool_calls:
                tool_fn = self.tool_map[tool_call['name']]
                tool_result = tool_fn.invoke(tool_call['args'])

                messages.append(ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call['id']
                ))

            response = self.llm.invoke(messages)
            messages.append(response)

        print_history(messages)
        return get_message_output(response)


class AgentSystem:
    def __init__(self, agents, llm):
        self.agents = agents
        self.history = []
        self.llm = llm

    def orchestrate(self, query):
        print(f'\nRunning Orchestrator...\n')
        
        agent_descriptions = '\n'.join(
            f'- {key}: {agent.description}'
            for key, agent in self.agents.items()
        )

        system_prompt = (
             'You are an orchestrator that decides which AI agents to use to answer a user query.\n\n'

            f'Available agents:\n{agent_descriptions}\n\n'

            'Given the user\'s query, return a JSON object with a single key "agents" containing a list of agent keys to run. The agents will then be run in that order.\n'
            'Only include agents that are relevant to the query. You may choose to repeat agents where necessary.\n'
            'Return ONLY the JSON object in a simple text form, no explanation.\n\n'

            'Example response:\n'
            f'{{"agents": {str(list(self.agents.keys())[:2]).replace('\'', '"')}}}'
        )

        messages = [SystemMessage(system_prompt), HumanMessage(query)]
        response = self.llm.invoke(messages)
        messages.append(response)

        print_history(messages)

        try:
            output = get_message_output(response)
            if output.startswith('```json') and output.endswith('```'):
                output = output[7:-3].strip()
            result = json.loads(output)
            return result['agents']

        except (json.JSONDecodeError, KeyError, TypeError):
            print('Routing failed, falling back to all agents.')
            return list(self.agents.keys())
        
    def synthesize(self, query, outputs):
        print(f'\nRunning Synthesizer...\n')

        system_prompt = (
            'You are a AI response synthesizer. '
            'You will be given responses from multiple specialist agents. '
            'Combine them into one clear, well-structured final synthesized response '
            'that addresses the user\'s original query.'
        )
        query_with_context = (
            f'User query:\n"{query}"\n\n'
            f'The agents have responded as follows.\n\n{outputs}'
            'Provide your final synthesized response.'
        )
        messages = [SystemMessage(system_prompt), HumanMessage(query_with_context)]
        response = self.llm.invoke(messages)
        messages.append(response)

        print_history(messages)        
        return get_message_output(response)
        
    def run_agents(self, query):
        agent_order = self.orchestrate(query)

        outputs = ''

        for agent_key in agent_order:
            if agent_key not in self.agents:
                print(f'Unknown agent {agent_key}, skipping.')
                continue
            
            agent = self.agents[agent_key]
            agent_order_str = ', '.join([self.agents[key].name for key in agent_order])

            if outputs:
                query_with_context = (
                    f'You are a member of an AI agent team working on answering the user query. The list of agents to be run is, in order: {agent_order_str}\n\n'
                    f'User query:\n"{query}"\n\n'
                    f'Previous agents have provided the following responses.\n\n{outputs}'
                    'Now provide your own contribution.'
                )
            else:
                query_with_context = (
                    f'You are a member of an AI agent team working on answering the user query. The list of agents to be run is, in order: {agent_order_str}\n\n'
                    f'User query:\n"{query}"\n\n'
                    'As the first agent to be run, provide your own contribution.'
                )

            output = agent.run(query_with_context)
            outputs += f'{agent.name}:\n"{output}"\n\n'

        synthesized_output = self.synthesize(query, outputs)

        return outputs, synthesized_output

In [64]:
def make_agents(agent_keys=None):
    agents = {}
    for key in (agent_keys or AGENT_REGISTRY.keys()):
        llm = make_llm(
            model=AGENT_REGISTRY[key].get('model'),
            temperature=AGENT_REGISTRY[key].get('temperature')
        )
        agents[key] = Agent(key, llm)

    return agents

system = AgentSystem(
    agents=make_agents(),
    llm=make_llm(SYSTEM_MODEL, SYSTEM_TEMPERATURE)
)

outputs, synthesized_output = system.run_agents('Help me plan a 10-day trip to Paris look for 2 people with a $5000 budget.')


Running Orchestrator...

SYSTEM
You are an orchestrator that decides which AI agents to use to answer a user query.

Available agents:
- travel_researcher: Finds attractions, activities, cultural info, and destination facts. Use for queries about what to do, see, or experience.
- budget_analyst: Handles all cost, currency, and financial planning. Use for queries involving money, budgets, exchange rates, or cost breakdowns.
- risk_advisor: Assesses travel safety, advisories, and risks. Use for queries about safety, travel warnings, or risk mitigation.

Given the user's query, return a JSON object with a single key "agents" containing a list of agent keys to run. The agents will then be run in that order.
Only include agents that are relevant to the query. You may choose to repeat agents where necessary.
Return ONLY the JSON object in a simple text form, no explanation.

Example response:
{"agents": ["travel_researcher", "budget_analyst"]}
-----------------------------------------------

In [65]:
print(synthesized_output)

Of course. Here is a synthesized plan for your 10-day trip to Paris, combining expert recommendations on attractions with a realistic budget analysis.

### Planning Your 10-Day Parisian Trip for Two on a $5,000 Budget

A 10-day trip to Paris for two is a wonderful goal. While a $5,000 budget is ambitious for this length of stay, it is achievable with careful planning and strategic choices. This guide combines a realistic look at the costs with recommendations for an unforgettable experience.

---

### **1. Budget Analysis: A Realistic Financial Overview**

First, let's look at a typical cost breakdown for a 10-day trip to Paris. This will help us understand where your budget will go and where we can make adjustments.

**Estimated Mid-Range Trip Cost (for 2 people):**

*   **Flights:** ~$1,600 ($800 per person, round trip)
*   **Accommodation:** ~$2,160 ($216/night for a mid-range hotel)
*   **Food & Dining:** ~$1,680 ($84 per person/day)
*   **Activities & Local Transport:** ~$1,200 ($